# LLM generate

In [ ]:
#|default_exp utils.llm

In [ ]:
#|export
import asyncio
import json

from ai_index.const import llm_models_config_path
from ai_index.utils._model_config import _load_model_config, _resolve_model_args, _split_remote_kwargs


def extract_json(text: str, validator=None) -> dict | None:
    """Extract a JSON object from text that may contain surrounding prose.

    Tries all pairs of '{' and '}' positions (opening: left-to-right,
    closing: right-to-left for each opening) and returns the first
    substring that parses as valid JSON and passes the optional validator.

    Args:
        text: Text potentially containing a JSON object.
        validator: Optional callable that takes a dict and raises on invalid
            data (e.g. a pydantic model_validate). If provided, only returns
            a parse that passes validation.

    Returns:
        Parsed dict, or None if no valid JSON found.
    """
    text = text.strip()
    close_positions = [i for i, c in enumerate(text) if c == '}']
    if not close_positions:
        return None
    pos = 0
    while True:
        idx = text.find("{", pos)
        if idx == -1:
            return None
        for close_pos in reversed(close_positions):
            if close_pos <= idx:
                break
            try:
                parsed = json.loads(text[idx:close_pos + 1])
                if not isinstance(parsed, dict):
                    continue
                if validator is not None:
                    validator(parsed)
                return parsed
            except Exception:
                continue
        pos = idx + 1


def is_reasoning_model(model_key: str) -> bool:
    """Check if a model emits a reasoning/thinking prefix before its answer."""
    _, cfg = _load_model_config(llm_models_config_path, model_key)
    return cfg["reasoning"]


def uses_structured_output(model_key: str) -> bool:
    """Check if a model supports vLLM structured output (json_schema)."""
    _, cfg = _load_model_config(llm_models_config_path, model_key)
    return cfg["structured_output"]

def llm_generate(
    prompts: list[str],
    *,
    model: str,
    **kwargs,
) -> list[str]:
    """Generate LLM responses using a named model key from llm_models.toml.

    The model key determines the execution mode (api/local/sbatch).
    Any explicit **kwargs override config values.

    Pass slurm_accounting={} to collect Slurm resource accounting data
    (sbatch mode only).
    """
    mode, model_name, cfg = _resolve_model_args(llm_models_config_path, model, kwargs)
    slurm_accounting = cfg.pop("slurm_accounting", None)

    if mode in ("api", "local"):
        from llm_runner.llm import run_llm_generate
        if mode == "api":
            cfg["backend"] = "api"
        return run_llm_generate(prompts, model_name=model_name, **cfg)

    elif mode == "sbatch":
        from isambard_utils.orchestrate import run_remote
        remote_kw = _split_remote_kwargs(cfg)
        cfg["model_name"] = model_name
        cfg["tensor_parallel_size"] = remote_kw["gpus"]
        result = run_remote(
            "llm_generate",
            inputs={"prompts": prompts},
            config_dict=cfg,
            required_models=[model_name],
            **remote_kw,
        )
        if slurm_accounting is not None and "_slurm_accounting" in result:
            slurm_accounting.update(result["_slurm_accounting"])
        return result["responses"]

    else:
        raise ValueError(f"Unknown mode: {mode!r}")

In [ ]:
#|export
async def allm_generate(
    prompts: list[str],
    *,
    model: str,
    **kwargs,
) -> list[str]:
    """Async version of llm_generate.

    For api/local modes, runs the sync run_llm_generate in a thread.
    For sbatch mode, uses the native async arun_remote.

    Pass slurm_accounting={} to collect Slurm resource accounting data
    (sbatch mode only).
    """
    mode, model_name, cfg = _resolve_model_args(llm_models_config_path, model, kwargs)
    slurm_accounting = cfg.pop("slurm_accounting", None)

    if mode in ("api", "local"):
        from llm_runner.llm import run_llm_generate
        if mode == "api":
            cfg["backend"] = "api"
        return await asyncio.to_thread(run_llm_generate, prompts, model_name=model_name, **cfg)

    elif mode == "sbatch":
        from isambard_utils.orchestrate import arun_remote
        remote_kw = _split_remote_kwargs(cfg)
        cfg["model_name"] = model_name
        cfg["tensor_parallel_size"] = remote_kw["gpus"]
        result = await arun_remote(
            "llm_generate",
            inputs={"prompts": prompts},
            config_dict=cfg,
            required_models=[model_name],
            **remote_kw,
        )
        if slurm_accounting is not None and "_slurm_accounting" in result:
            slurm_accounting.update(result["_slurm_accounting"])
        return result["responses"]

    else:
        raise ValueError(f"Unknown mode: {mode!r}")